# GIAI ĐOẠN 3:EDA(Phân tích khám phá)
Giai đoạn này nhằm biến dữ liệu sạch thành Insight(hiểu biết trực quan), giải thích tại sao giá nhà lại khác nhau.<div>

Tổng quan các bước: 
1. Phân tích Phân phối(Distribution & New Features)
2. Phân tích Vị trí & Đa biến (Locational & Multivariate)
3. Phân tích Tương quan & Tương tác (Correlation & Interaction)
4. Phân tích Phân phối Biến Binary & Categorical

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# --- Cài đặt ban đầu ---
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

try:
    df = pd.read_csv('../data/processed/extracted_data.csv')
except FileNotFoundError:
    print("Error: Cleaned data file not found.")
    
# --- 1. Định nghĩa các biến cần phân tích ---
features_to_plot = ['price', 'area']#, 'price_per_sqm', 'total_utility_cost','amenity_score'
n_features = len(features_to_plot)

# --- 2. Tạo bảng biểu đồ (2 Hàng x 5 Cột) ---
# Hàng 1: Histogram (Phân phối) | Hàng 2: Box Plot (Outlier)
fig, axes = plt.subplots(2, n_features, figsize=(20, 8))
plt.suptitle('B1: Phân tích Phân phối Biến Số Định lượng', fontsize=16, y=1.02)


for i, col in enumerate(features_to_plot):
    # Lấy dữ liệu
    data_to_plot = df[col].dropna()
    # Hàng 1 Histogram (Kiểm tra Skewness) 
    
    is_skewed = data_to_plot.skew() > 1.0 # Kiểm tra nếu độ lệch lớn hơn 1 (lệch phải đáng kể)
    
    if is_skewed:
        # Sử dụng log1p (log(1+x)) cho các biến lệch phải để phân phối dễ nhìn hơn
        log_data = np.log1p(data_to_plot)
        axes[0, i].hist(log_data, bins=30, edgecolor='black', alpha=0.7)
        axes[0, i].set_title(f'Histogram: Log(1 + {col.replace("_", " ").title()})', fontsize=10)
        axes[0, i].set_xlabel('Log Value')
    else:
        # Sử dụng thang tuyến tính thông thường
        axes[0, i].hist(data_to_plot, bins=30, edgecolor='black', alpha=0.7)
        axes[0, i].set_title(f'Histogram: {col.replace("_", " ").title()}', fontsize=10)
        axes[0, i].set_xlabel('Value')
        
    axes[0, i].set_ylabel('Frequency')
    
    skewness_val = data_to_plot.skew()
    
    axes[0, i].text(0.95, 0.95, f'Skew: {skewness_val:.2f}', 
                    transform=axes[0, i].transAxes, 
                    fontsize=9, 
                    verticalalignment='top', 
                    horizontalalignment='right', 
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.5))

    # --- Hàng 2: Box Plot (Kiểm tra Outlier còn sót) ---
    sns.boxplot(y=data_to_plot, ax=axes[1, i], color=sns.color_palette("husl", n_features)[i])
    axes[1, i].set_title(f'Box Plot: {col.replace("_", " ").title()}', fontsize=10)
    axes[1, i].set_ylabel('')
    axes[1, i].tick_params(axis='x', labelbottom=False)
    
plt.tight_layout(rect=[0, 0, 1, 1.0])
plt.savefig('B1_numerical_distribution_analysis.png')
print("Đã tạo 'B1_numerical_distribution_analysis.png' để trực quan hóa phân phối và kiểm tra Outlier/Skewness.")